In [ ]:
import os  
import json  
import gzip  
from typing import Dict, List  
from collections import OrderedDict  

import pandas as pd  
from rdkit import Chem  
from rdkit.Chem import AllChem  
from rdkit import DataStructs  

In [ ]:
import re                                   
import pubchempy as pcp                     

def from_smiles_get_iupac_and_cas(smiles: str):
    
    
    if smiles is None or not isinstance(smiles, str) or smiles.strip() == "":
        return "", ""                        

    try:
        
        compounds = pcp.get_compounds(smiles, namespace='smiles')
    except Exception as e:
        
        print("Public message removed for release.")
        return "", ""

    
    if not compounds:
        return "", ""

    
    compound = compounds[0]

    
    iupac_name = getattr(compound, "iupac_name", None) or ""

    
    cas_rn = ""

    
    try:
        synonyms = compound.synonyms or []   
    except Exception:
        synonyms = []                        

    
    if not synonyms:
        try:
            
            syn_data = pcp.get_synonyms(compound.cid, namespace='cid')
            
            if syn_data and isinstance(syn_data, list):
                for item in syn_data:
                    
                    if isinstance(item, dict):
                        if 'Synonym' in item and isinstance(item['Synonym'], list):
                            synonyms = item['Synonym']
                            break
                        if 'synonyms' in item and isinstance(item['synonyms'], list):
                            synonyms = item['synonyms']
                            break
        except Exception:
            pass                             

    
    cas_pattern = re.compile(r'\b\d{2,7}-\d{2}-\d\b')

    
    for s in synonyms or []:
        if not isinstance(s, str):
            continue                         
        m = cas_pattern.search(s)            
        if m:
            cas_rn = m.group(0)              
            break                            

    
    return iupac_name, cas_rn

In [ ]:
from typing import Tuple, Dict
import pandas as pd  

def annotate_name_cas_from_smiles(
    df: pd.DataFrame,
    smiles_col: str = "SMILES",
    name_col: str = "Name",
    cas_col: str = "CAS"
) -> pd.DataFrame:
    
    df = df.copy()  

    
    if smiles_col not in df.columns:
        raise ValueError("Public message removed for release.")

    
    if name_col not in df.columns:
        df[name_col] = ""  
    if cas_col not in df.columns:
        df[cas_col] = ""   

    
    cache: Dict[str, Tuple[str, str]] = {}

    
    for idx in df.index:
        raw = df.at[idx, smiles_col]                 
        smi = "" if pd.isna(raw) else str(raw).strip()  

        if smi == "":                                
            continue

        
        if smi in cache:
            name, cas = cache[smi]                   
        else:
            try:
                
                name, cas = from_smiles_get_iupac_and_cas(smi)
            except Exception as e:
                
                print("Public message removed for release.")
                name, cas = "", ""
            cache[smi] = (name, cas)                 

        
        if isinstance(name, str) and name != "":
            df.at[idx, name_col] = name              
        if isinstance(cas, str) and cas != "":
            df.at[idx, cas_col] = cas                

    return df  

In [ ]:

df = pd.read_excel("processed_file.xlsx")
df = annotate_name_cas_from_smiles(df)
df.to_excel("processed_file.xlsx", index=None)

In [ ]:
df